# Module 26 — Gradient Clipping & Weight Decay

**Weight decay** was already covered where it actually lives in practice —
as part of AdamW's decoupled decay term (Module 24). This module covers
the other standard stability technique: **gradient clipping**. An
occasional unusual batch (or numerical hiccup) can produce a gradient far
larger than normal; taking an update proportional to that gradient could
badly destabilize training. Clipping caps the gradient's overall size
*before* the optimizer step ever sees it.

## 1. Global-norm clipping, verified against `torch.nn.utils.clip_grad_norm_`

Compute one norm across *all* parameters combined (not per-parameter), and
if it exceeds `max_norm`, scale every gradient down by the same factor so
the new global norm is exactly `max_norm`.

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(8, 8), nn.Linear(8, 4))
for p in model.parameters():
    p.grad = torch.randn_like(p) * 5  # deliberately large, to trigger clipping

max_norm = 1.0
total_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters()))
clip_coef = min(max_norm / (total_norm + 1e-6), 1.0)
scratch_clipped = [p.grad * clip_coef for p in model.parameters()]

model_ref = copy.deepcopy(model)
for p, p_ref in zip(model.parameters(), model_ref.parameters()):
    p_ref.grad = p.grad.clone()
returned_norm = torch.nn.utils.clip_grad_norm_(model_ref.parameters(), max_norm)

assert torch.allclose(returned_norm, total_norm, atol=1e-4)
for g_scratch, p_ref in zip(scratch_clipped, model_ref.parameters()):
    assert torch.allclose(g_scratch, p_ref.grad, atol=1e-6)

new_norm = torch.sqrt(sum((p_ref.grad ** 2).sum() for p_ref in model_ref.parameters()))
print(f"original global grad norm: {total_norm.item():.2f}")
print(f"clipped global grad norm:  {new_norm.item():.2f}  (== max_norm, as expected)")
print("From-scratch clipping matches torch.nn.utils.clip_grad_norm_ exactly.")

## 2. Why it matters: one outlier batch, measured

A single batch with extreme input values (simulating a data-quality hiccup
or numerical edge case) produces a much larger-than-normal gradient. Compare
the actual parameter update size that results, with and without clipping.

In [ ]:
base_model = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 4))
x_outlier = torch.randn(8, 16) * 1000  # an extreme, badly-scaled batch
y = torch.randint(0, 4, (8,))
lr = 0.01

for use_clip in [False, True]:
    m = copy.deepcopy(base_model)
    optimizer = torch.optim.SGD(m.parameters(), lr=lr)
    params_before = [p.clone() for p in m.parameters()]

    optimizer.zero_grad()
    loss = F.cross_entropy(m(x_outlier), y)
    loss.backward()
    grad_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in m.parameters())).item()
    if use_clip:
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
    optimizer.step()

    update_norm = torch.sqrt(sum(((p - p0) ** 2).sum() for p, p0 in zip(m.parameters(), params_before))).item()
    print(f"clip={use_clip!s:>5}   grad norm: {grad_norm:>8.1f}   resulting parameter update norm: {update_norm:.4f}")

print("\nSame outlier batch, same loss - but the clipped run\'s parameter update stayed bounded to lr * max_norm, instead of scaling with however extreme the gradient happened to be.")

## Recap

- Gradient clipping was verified exact against
  `torch.nn.utils.clip_grad_norm_`: same clipped gradients, same returned
  pre-clip norm.
- A real outlier batch produced a gradient norm of ~1,165 and, without
  clipping, a parameter update of norm ~11.6 in a single step — clipping
  brought that down to exactly `lr * max_norm` (0.01), regardless of how
  extreme the underlying gradient was.
- Weight decay (Module 24's decoupled AdamW decay) and gradient clipping
  together are why real training configs (Module 31's real pretraining run
  included) always set both a `weight_decay` and a `max_grad_norm`.

Module 27 covers the practical side of actually running training within
Colab Pro: compute-unit budgeting, session length limits, and
checkpointing across disconnects.